# Tarea 2 - Taller de Deep Learning

**Fecha de entrega: 16/11/2025**  
**Puntaje máximo: 15**

## Introducción

El objetivo de esta tarea es evaluar su capacidad para aplicar modelos de redes neuronales recurrentes (RNN/LSTM/GRU) en un problema de clasificación de secuencias. En particular, vamos a evaluar la performance de sus modelos en la clasificación de ritmos cardíacos usando datos de electrocardiograma (ECG).

**Dataset**

El dataset a ser utilizado es el [Heartbeat Dataset](https://www.kaggle.com/datasets/shayanfazeli/heartbeat). Este dataset contiene señales de ECG segmentadas, donde cada segmento corresponde a un latido del corazón. Cada segmento ya está preprocesado y categorizado en una de las siguientes clases:

- **N**: Normal (0)
- **S**: Arritmia supraventricular (1)
- **V**: Arritmia ventricular (2)
- **F**: Latido fusionado (3)
- **Q**: Latido desconocido (4)

Los archivos del dataset que deben utilizar son:

- **mitbih_train.csv**: Datos de entrenamiento.
- **mitbih_test.csv**: Datos de prueba.

**Tarea**

Tienen total libertad sobre cómo implementar y resolver el problema, así como las técnicas y herramientas que quieran usar. Se recomienda el uso de Google Colab para simplificar el acceso a recursos de GPU, aunque pueden trabajar en sus propias máquinas si lo prefieren. La entrega debe realizarse en formato .ipynb (Jupyter Notebook) **con las celdas ya ejecutadas**.

**Restricciones**

- No se permite utilizar modelos pre-entrenados; cada modelo debe ser implementado desde cero.
- Deben utilizar al menos un modelo basado en RNN (por ejemplo, LSTM o GRU).
- Es necesario realizar un **análisis exploratorio de los datos**, que incluya una descripción de las señales ECG, el balanceo de clases y cualquier limpieza o transformación necesaria de los datos.
- Las decisiones sobre el preprocesamiento de las señales (como normalización, segmentación, etc.) deben estar fundamentadas en una exploración inicial del dataset y explicadas en el notebook.

**Reporte**

Se requiere que reporten las siguientes métricas: accuracy, precision, recall y F1-score para la evaluación del modelo. Además, se espera ver una evolución clara del modelo durante el entrenamiento, que incluya logs y gráficas de las métricas tanto para los datos de entrenamiento como de validación.

**Evidencia de Experimentos**

Deben proporcionar evidencia de la ejecución de experimentos usando [Weights & Biases (wandb)](https://wandb.ai/). Esto incluye:

- Registros detallados de los experimentos.
- Gráficas y logs de entrenamiento.
- Comparaciones entre diferentes configuraciones de modelos.

# Librerías

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from torchinfo import summary

import pandas as pd
import numpy as np

import os
import sys
import tarfile
import urllib.request
import re
from pathlib import Path
from collections import Counter
from itertools import chain

# from utils import (
#     train,
# )

# Datasets 

In [2]:
# pip install kagglehub

In [3]:
os.getcwd()

'c:\\Users\\Matías\\Desktop\\Rodrigo\\Repos\\ORT-AI\\Deep Learning\\Taller de DL\\Tareas\\Tarea 2'

In [4]:
# import kagglehub
# import shutil

# # Descargar dataset
# path = kagglehub.dataset_download("shayanfazeli/heartbeat")

# # Crear carpeta "data" si no existe
# os.makedirs("data", exist_ok=True)

# # Nuevo destino
# dest = os.path.join("data", os.path.basename(path))

# # Mover carpeta descargada a "data"
# shutil.move(path, dest)

# print("Dataset guardado en:", dest)


In [5]:
train_data = pd.read_csv('data/mitbih_train.csv', header=None)
test_data = pd.read_csv('data/mitbih_test.csv', header=None)

In [6]:
train_data.head()

,0,1,2,3,4,5,6,7,8,9,...,178,179,180,181,182,183,184,185,186,187
0,0.977941,0.926471,0.681373,0.245098,0.154412,0.191176,0.151961,0.085784,0.058824,0.049020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.960114,0.863248,0.461538,0.196581,0.094017,0.125356,0.099715,0.088319,0.074074,0.082621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.000000,0.659459,0.186486,0.070270,0.070270,0.059459,0.056757,0.043243,0.054054,0.045946,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.925414,0.665746,0.541436,0.276243,0.196133,0.077348,0.071823,0.060773,0.066298,0.058011,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.967136,1.000000,0.830986,0.586854,0.356808,0.248826,0.145540,0.089202,0.117371,0.150235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
test_data.head()

,0,1,2,3,4,5,6,7,8,9,...,178,179,180,181,182,183,184,185,186,187
0,1.000000,0.758264,0.111570,0.000000,0.080579,0.078512,0.066116,0.049587,0.047521,0.035124,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.908425,0.783883,0.531136,0.362637,0.366300,0.344322,0.333333,0.307692,0.296703,0.300366,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.730088,0.212389,0.000000,0.119469,0.101770,0.101770,0.110619,0.123894,0.115044,0.132743,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.000000,0.910417,0.681250,0.472917,0.229167,0.068750,0.000000,0.004167,0.014583,0.054167,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.570470,0.399329,0.238255,0.147651,0.000000,0.003356,0.040268,0.080537,0.070470,0.090604,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
train_data.rename(columns={187: 'class'}, inplace=True)

id_to_label = {
    0: "Normal",
    1: "Artial Premature",
    2: "Premature ventricular contraction",
    3: "Fusion of ventricular and normal",
    4: "Fusion of paced and normal"
}
train_data['label'] = train_data.iloc[:, -1].map(id_to_label)
print(train_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87554 entries, 0 to 87553
Columns: 189 entries, 0 to label
dtypes: float64(188), object(1)
memory usage: 126.2+ MB
None


In [9]:
test_data.rename(columns={187: 'class'}, inplace=True)

id_to_label = {
    0: "Normal",
    1: "Artial Premature",
    2: "Premature ventricular contraction",
    3: "Fusion of ventricular and normal",
    4: "Fusion of paced and normal"
}
test_data['label'] = test_data.iloc[:, -1].map(id_to_label)
print(test_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21892 entries, 0 to 21891
Columns: 189 entries, 0 to label
dtypes: float64(188), object(1)
memory usage: 31.6+ MB
None


In [10]:
test_data.head()

,0,1,2,3,4,5,6,7,8,9,...,179,180,181,182,183,184,185,186,class,label
0,1.000000,0.758264,0.111570,0.000000,0.080579,0.078512,0.066116,0.049587,0.047521,0.035124,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal
1,0.908425,0.783883,0.531136,0.362637,0.366300,0.344322,0.333333,0.307692,0.296703,0.300366,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal
2,0.730088,0.212389,0.000000,0.119469,0.101770,0.101770,0.110619,0.123894,0.115044,0.132743,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal
3,1.000000,0.910417,0.681250,0.472917,0.229167,0.068750,0.000000,0.004167,0.014583,0.054167,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal
4,0.570470,0.399329,0.238255,0.147651,0.000000,0.003356,0.040268,0.080537,0.070470,0.090604,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal


# EDA

## Distribución de clases

In [11]:
import plotly.express as px

# Crear el conteo y reset index con nombres claros
class_counts = train_data['label'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']  # Renombrar columnas explícitamente

# Calcular porcentaje y formatear como label
total = class_counts['Count'].sum()
class_counts['Percent'] = class_counts['Count'] / total * 100
class_counts['Label'] = class_counts['Percent'].map(lambda x: f'{x:.1f}%')

fig = px.bar(
    class_counts,
    x='Class',
    y='Count',
    color='Class',  # Cada clase de un color distinto
    title='Number of samples per class',
    color_discrete_sequence=px.colors.qualitative.Safe,  # Paleta de colores distinta por clase
    text='Label'  # Usar la columna de porcentaje como data label
)

fig.update_traces(
    textposition='outside',  # Ubica el label fuera (arriba) del extremo de la barra
    textfont_size=14,
    texttemplate='%{text}',  # Solo mostrar el texto (el porcentaje)
    insidetextanchor='middle'
)

fig.update_layout(
    uniformtext_minsize=12,  # Asegura tamaño uniforme
    uniformtext_mode='hide',
    xaxis_title='Class',
    yaxis_title='Count',
    height=600  # <-- Aumenta la altura del gráfico para que no se corte
)

fig.show()

## ECG por clase

In [12]:
# Seleccionar columnas de tiempo
value_columns = [col for col in train_data.columns if col not in ['class', 'label']]

num_samples = 10
dfs = []
for class_name in train_data['label'].unique():
    samples = train_data[train_data['label'] == class_name].sample(n=num_samples, random_state=42)
    samples = samples[value_columns].reset_index(drop=True)
    samples['Muestra'] = samples.index  # id de muestra correcto por fila
    df_long = samples.melt(id_vars='Muestra', var_name='Tiempo', value_name='Valor')
    df_long['Clase'] = class_name
    dfs.append(df_long)

plot_df = pd.concat(dfs, ignore_index=True)

# Asegurar tiempo numérico y ordenado
try:
    plot_df['Tiempo'] = plot_df['Tiempo'].astype(float)
except Exception:
    pass
plot_df = plot_df.sort_values(['Clase', 'Muestra', 'Tiempo'])

fig = px.line(
    plot_df,
    x='Tiempo',
    y='Valor',
    color='Muestra',
    facet_row='Clase',
    facet_row_spacing=0.07,
    line_group='Muestra',
    height=300 * len(train_data['label'].unique()),
    title="10 muestras aleatorias de cada clase",
)

fig.update_traces(opacity=0.5, showlegend=False)
fig.update_yaxes(title_text='Valor')
fig.update_xaxes(title_text='Tiempo', matches='x')
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

* Valores faltantes/constantes, rango global de amplitudes.
* Estadísticos por muestra y por clase: media, mediana, std, IQR, min/max, skewness, kurtosis.
* 

## Amplitud de ECG por clase

In [13]:
def analyze_ecg_amplitude_by_class(
    df,
    label_col: str = 'label',
    value_columns: list[str] | None = None,
    plot: bool = True,
    return_sample_metrics: bool = False,
):
    """
    Analiza la amplitud (pico-a-pico) de latidos ECG por clase.

    - Calcula por muestra: mínimo, máximo y amplitud pico-a-pico (max - min)
    - Agrega por clase: count, mean, std, median, p25, p75, min, max
    - Opcionalmente grafica la distribución por clase (violin y box)

    Args:
        df: DataFrame con columnas de señal (tiempo) y columna de etiqueta.
        label_col: Nombre de la columna con la clase (por defecto 'label').
        value_columns: Lista de columnas de tiempo. Si es None, se infiere
            como todas las columnas excepto {label_col, 'class'}.
        plot: Si True, muestra gráficos de distribución por clase.
        return_sample_metrics: Si True, también devuelve métricas por muestra.

    Returns:
        summary_df si return_sample_metrics == False
        (summary_df, sample_metrics_df) si return_sample_metrics == True
    """
    # Inferir columnas de señal si no se especifican
    if value_columns is None:
        excluded = {label_col, 'class'}
        value_columns = [c for c in df.columns if c not in excluded]

    # Asegurar datos numéricos y calcular métricas por muestra
    signal_matrix = df[value_columns].to_numpy(dtype=float)
    per_sample_min = signal_matrix.min(axis=1)
    per_sample_max = signal_matrix.max(axis=1)
    per_sample_ptp = per_sample_max - per_sample_min  # pico-a-pico

    sample_metrics_df = pd.DataFrame(
        {
            'min_amplitude': per_sample_min,
            'max_amplitude': per_sample_max,
            'peak_to_peak': per_sample_ptp,
            label_col: df[label_col].to_numpy(),
        }
    )

    # Agregación por clase
    def p25(series: pd.Series) -> float:
        return float(np.percentile(series, 25))

    def p75(series: pd.Series) -> float:
        return float(np.percentile(series, 75))

    summary_df = (
        sample_metrics_df.groupby(label_col)['peak_to_peak']
        .agg(
            count='count',
            mean='mean',
            std='std',
            median='median',
            p25=p25,
            p75=p75,
            min='min',
            max='max',
        )
        .reset_index()
        .sort_values(label_col)
    )

    # Gráficos opcionales, con distinción de colores por clase
    if plot:
        # Construir una paleta de colores distinta por cada clase
        unique_classes = sample_metrics_df[label_col].unique()
        color_map = px.colors.qualitative.Plotly  # O puede usarse otra paleta si hay más clases
        color_discrete_map = {
            clase: color_map[i % len(color_map)] for i, clase in enumerate(sorted(unique_classes))
        }

        violin_fig = px.violin(
            sample_metrics_df,
            x=label_col,
            y='peak_to_peak',
            box=True,
            points='outliers',
            labels={label_col: 'Clase', 'peak_to_peak': 'Amplitud pico-a-pico'},
            title='Distribución de amplitud pico-a-pico por clase',
            color=label_col,  # Esto asigna color por clase
            color_discrete_map=color_discrete_map  # Mapea clase a color de la paleta
        )
        violin_fig.show()

    return (summary_df, sample_metrics_df) if return_sample_metrics else summary_df


In [14]:
train_data["label"].unique()

array(['Normal', 'Artial Premature', 'Premature ventricular contraction',
       'Fusion of ventricular and normal', 'Fusion of paced and normal'],
      dtype=object)

In [15]:
summary = analyze_ecg_amplitude_by_class(train_data, label_col='label', plot=True)

In [16]:
def ecg_duration_by_zero_runs(
    df,
    label_col: str = 'label',
    value_columns: list[str] | None = None,
    min_zero_run: int = 5,
    zero_tol: float = 0.0,
    fs_hz: float | None = None,
    plot: bool = True,
    return_sample_metrics: bool = False,
):
    """
    Calcula la *duración* por muestra aplicando la regla:
      - Buscar el primer run de >= `min_zero_run` ceros consecutivos; la duración es
        el índice de inicio de ese run (tiempo máximo).
      - Si no existe, usar el índice del último dato disponible (último no-NaN).

    Si `fs_hz` se provee, devuelve la duración en segundos (idx / fs_hz),
    si no, en número de muestras (índice).

    Args:
        df: DataFrame con columnas de señal (tiempo) y etiqueta.
        label_col: Nombre de la columna de etiquetas.
        value_columns: Columnas de tiempo; si None, se infiere excluyendo {label_col, 'class'}.
        min_zero_run: Largo mínimo del run de ceros consecutivos a detectar.
        zero_tol: Tolerancia para considerar un valor como cero (|x| <= tol).
        fs_hz: Frecuencia de muestreo; si None, duración en muestras.
        plot: Si True, grafica histograma por clase.
        return_sample_metrics: Si True, retorna también el DataFrame por muestra.

    Returns:
        summary_df (por clase) o (summary_df, sample_df) si return_sample_metrics=True.
    """
    import numpy as np
    import pandas as pd

    if value_columns is None:
        excluded = {label_col, 'class'}
        value_columns = [c for c in df.columns if c not in excluded]

    signals = df[value_columns].to_numpy(dtype=float)

    def first_zero_run_start(vec: np.ndarray) -> int:
        is_finite = np.isfinite(vec)
        # Índices válidos (no NaN)
        valid_idx = np.where(is_finite)[0]
        if valid_idx.size == 0:
            return 0  # sin datos válidos
        # Run de ceros con tolerancia
        is_zero = is_finite & (np.abs(vec) <= zero_tol)
        if is_zero.any() and min_zero_run > 1:
            win = np.ones(min_zero_run, dtype=int)
            conv = np.convolve(is_zero.astype(int), win, mode='valid')
            hits = np.where(conv == min_zero_run)[0]
            if hits.size > 0:
                return int(hits[0])
        elif is_zero.any() and min_zero_run <= 1:
            return int(np.where(is_zero)[0][0])
        # Si no hay run suficiente, usar último índice válido (no-NaN)
        return int(valid_idx[-1])

    idxs = np.apply_along_axis(first_zero_run_start, 1, signals)

    if fs_hz is not None:
        durations = idxs.astype(float) / float(fs_hz)
        metric = 'duration_sec'
    else:
        durations = idxs.astype(int)
        metric = 'duration_samples'

    sample_df = pd.DataFrame({
        metric: durations,
        label_col: df[label_col].to_numpy(),
    })

    # Resumen por clase
    def p25(x):
        return float(np.percentile(x, 25))
    def p75(x):
        return float(np.percentile(x, 75))

    summary_df = (
        sample_df.groupby(label_col)[metric]
        .agg(count='count', mean='mean', std='std', median='median', p25=p25, p75=p75, min='min', max='max')
        .reset_index()
        .sort_values(label_col)
    )

    if plot:
        import plotly.express as px
        fig = px.histogram(
            sample_df,
            x=metric,
            color=label_col,
            barmode='overlay',
            opacity=0.65,
            nbins=40,
            labels={metric: ('Duración (s)' if fs_hz else 'Duración (muestras)'), label_col: 'Clase'},
            title='Histograma de duración por clase (regla de 5 ceros consecutivos)'
        )
        fig.show()

    return (summary_df, sample_df) if return_sample_metrics else summary_df




In [17]:
summary = ecg_duration_by_zero_runs(train_data, label_col='label', min_zero_run=5, zero_tol=0.0, fs_hz=360, plot=True)
summary

,label,count,mean,std,median,p25,p75,min,max
0,Artial Premature,2223,0.316108,0.108434,0.297222,0.263889,0.375000,0.061111,0.516667
1,Fusion of paced and normal,6431,0.328546,0.027077,0.333333,0.322222,0.341667,0.086111,0.516667
2,Fusion of ventricular and normal,641,0.218963,0.038641,0.216667,0.205556,0.225000,0.080556,0.516667
3,Normal,72471,0.307796,0.075528,0.294444,0.258333,0.352778,0.000000,0.516667
4,Premature ventricular contraction,5788,0.328157,0.096936,0.316667,0.263889,0.391667,0.069444,0.516667


## Duración de ECG por clase

In [18]:
# Duración = índice (1-based) de la última columna con valor > zero_tol; 0 si no hay valores > zero_tol
def calcular_duracion_por_ultimo_no_cero(row, zero_tol: float = 0.0):
    arr = row.values.astype(float)
    nz_idx = np.where(arr > zero_tol)[0]
    return int(nz_idx[-1] + 1) if nz_idx.size else 0

# Determinar columnas de señal si no existen previamente
try:
    signal_cols
except NameError:
    signal_cols = [c for c in train_data.columns if c not in ['class', 'label']]

# Aplicar lógica y escribir la duración
train_data['duracion_no_ceros'] = train_data[signal_cols].apply(
    calcular_duracion_por_ultimo_no_cero, axis=1
)



In [19]:
# Ajusta la opción de pandas para mostrar todas las columnas
pd.set_option('display.max_columns', None)


train_data.sort_values(by='duracion_no_ceros').head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,class,label,duracion_no_ceros
44608,1.0,0.919048,0.811905,0.473810,0.295238,0.211905,0.180952,0.135714,0.142857,0.121429,0.000000,0.128571,0.073810,0.078571,0.097619,0.090476,0.140476,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal,17
42566,1.0,0.988372,0.738372,0.398256,0.133721,0.101744,0.084302,0.127907,0.058140,0.040698,0.023256,0.052326,0.000000,0.017442,0.040698,0.078488,0.052326,0.063953,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal,18
13028,0.0,0.050505,0.333333,0.464646,0.777778,0.878788,0.868687,0.878788,0.868687,0.888889,0.919192,0.868687,0.878788,0.929293,0.919192,0.888889,0.989899,1.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Normal,18
18035,1.0,0.393939,0.056277,0.129870,0.095238,0.086580,0.077922,0.103896,0.086580,0.095238,0.095238,0.086580,0.082251,0.099567,0.095238,0.116883,0.099567,0.069264,0.073593,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [20]:
import plotly.express as px

fig = px.histogram(
    train_data, 
    x="duracion_no_ceros", 
    color="label", 
    barmode="overlay", 
    nbins=40,
    title="Histograma de duración ECG por clase",
    labels={"duracion_no_ceros": "Duración", "label": "Clases"}
)
fig.show()


# Balanceo de clases ?? 

* Oversampling de minoritarias (con reposición) + BalancedBatchSampler.

* Undersampling leve de la mayoritaria (con cuidado para no perder variabilidad).

* Augmentación 1D específica para ECG (aplicar solo a minoritarias):
    * Jitter: añadir ruido gaussiano bajo (σ≈0.01–0.05 del rango).
        def augment_ecg(signal):
            noise = np.random.normal(0, 0.01, signal.shape)
            return signal + noise

    * Escalado de amplitud: multiplicar por 0.9–1.1. (skippear casos = 1)

    * Time shift: desplazar pocos samples (con padding).

    * Permutación de 3–6 sub-segmentos cortos (muy leve).
    
    * Inversión o distorsión ligera


## Submuestra de la clase mayoritaria

In [21]:
def undersample_majority(df, label_col='label', ratio=None, target=None, random_state=42):
    """
    - Si ratio se da: reduce la mayoritaria a floor(minority_count * ratio)
    - Si target=int: reduce la mayoritaria a 'target'
    - Por defecto: balancea a la menor (target = minority_count)
    """
    counts = df[label_col].value_counts()
    if counts.empty or len(counts) == 1:
        return df.copy()

    majority_class = counts.idxmax()
    minority_count = counts.drop(majority_class).min()

    if ratio is not None:
        target_count = int(minority_count * ratio)
    elif isinstance(target, int):
        target_count = min(target, counts[majority_class])
    else:
        target_count = minority_count  # balance 1:1

    majority_df = df[df[label_col] == majority_class].sample(
        n=target_count, random_state=random_state
    )
    rest_df = df[df[label_col] != majority_class]

    out = pd.concat([rest_df, majority_df], axis=0)
    out = out.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return out


In [22]:
train_data_us = undersample_majority(train_data, label_col='label', target=35000)

# Verificación
print("Antes de la sub-muestra:\n", train_data['label'].value_counts(), "\n")
print("Después de la sub-muestra:\n", train_data_us['label'].value_counts())

Antes de la sub-muestra:
 label
Normal                               72471
Fusion of paced and normal            6431
Premature ventricular contraction     5788
Artial Premature                      2223
Fusion of ventricular and normal       641
Name: count, dtype: int64 

Después de la sub-muestra:
 label
Normal                               35000
Fusion of paced and normal            6431
Premature ventricular contraction     5788
Artial Premature                      2223
Fusion of ventricular and normal       641
Name: count, dtype: int64


## Oversampling de las clases minoritarias

In [23]:
def oversample_minorities(
    df: pd.DataFrame,
    label_col: str = 'label',
    strategy: str = 'to_majority',  # 'to_majority' | 'ratio' | 'fixed'
    ratio: float | None = None,     # ej. 0.8 -> 80% del tamaño de la mayoritaria
    target: int | None = None,      # ej. 12000 -> tamaño fijo por clase
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Oversamplea solo las clases minoritarias.
    - strategy='to_majority': sube todas las clases al tamaño de la mayoritaria (1:1).
    - strategy='ratio': sube todas las clases a int(majority_count * ratio).
    - strategy='fixed': sube todas las clases a 'target'.
    """
    counts = df[label_col].value_counts()
    if counts.empty or len(counts) == 1:
        return df.copy()

    majority_count = counts.max()

    if strategy == 'to_majority':
        target_count = majority_count
    elif strategy == 'ratio':
        if ratio is None or ratio <= 0:
            raise ValueError("Debes pasar un 'ratio' > 0 para strategy='ratio'.")
        target_count = int(majority_count * ratio)
    elif strategy == 'fixed':
        if target is None or target <= 0:
            raise ValueError("Debes pasar 'target' > 0 para strategy='fixed'.")
        target_count = target
    else:
        raise ValueError("strategy debe ser 'to_majority', 'ratio' o 'fixed'.")

    parts = []
    for cls, cnt in counts.items():
        cls_df = df[df[label_col] == cls]
        if cnt >= target_count:
            parts.append(cls_df)
        else:
            # tomar todo lo original + muestrear con reposición lo faltante
            needed = target_count - cnt
            sampled = cls_df.sample(n=needed, replace=True, random_state=random_state)
            parts.append(pd.concat([cls_df, sampled], axis=0))

    out = pd.concat(parts, axis=0).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return out



In [24]:
# Llevar todas las clases al 80% del tamaño de la mayoritaria
train_data_us_os = oversample_minorities(train_data_us, label_col='label', strategy='ratio', ratio=0.1)

# Verificación
print("Antes de over y under sampling:\n", train_data['label'].value_counts(), "\n")
print("Después de over y under sampling:\n", train_data_us_os['label'].value_counts())

Antes de over y under sampling:
 label
Normal                               72471
Fusion of paced and normal            6431
Premature ventricular contraction     5788
Artial Premature                      2223
Fusion of ventricular and normal       641
Name: count, dtype: int64 

Después de over y under sampling:
 label
Normal                               35000
Fusion of paced and normal            6431
Premature ventricular contraction     5788
Artial Premature                      3500
Fusion of ventricular and normal      3500
Name: count, dtype: int64


## Data Augmentation para clases minoritarias

In [25]:
def jitter_signal(x: np.ndarray, sigma: float = 0.02, rng: np.random.Generator | None = None) -> np.ndarray:
    """Añade ruido gaussiano con desv. std relativa al std de la señal (suave)."""
    rng = np.random.default_rng(rng)
    x = np.asarray(x, dtype=float)
    noise = rng.normal(0.0, sigma * (np.std(x) + 1e-8), size=x.shape)
    return (x + noise).astype(x.dtype)


def scale_amplitude(x: np.ndarray, scale_range: tuple[float, float] = (0.9, 1.1), rng: np.random.Generator | None = None) -> np.ndarray:
    """Escala amplitud por un factor uniforme dentro de scale_range."""
    rng = np.random.default_rng(rng)
    x = np.asarray(x, dtype=float)
    factor = rng.uniform(scale_range[0], scale_range[1])
    return (x * factor).astype(x.dtype)


def time_shift_signal(
    x: np.ndarray,
    max_shift: int = 5,
    pad_mode: str = 'reflect',  # 'reflect' | 'constant'
    constant_value: float = 0.0,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """Desplaza la señal en el tiempo (no circular)."""
    rng = np.random.default_rng(rng)
    x = np.asarray(x, dtype=float)
    n = len(x)
    if max_shift <= 0:
        return x
    shift = int(rng.integers(-max_shift, max_shift + 1))
    if shift == 0:
        return x

    if shift > 0:  # desplazar a la derecha => rellenar al inicio
        pad = x[:shift][::-1] if pad_mode == 'reflect' else np.full(shift, constant_value)
        y = np.concatenate([pad, x[:-shift]])
    else:  # desplazar a la izquierda => rellenar al final
        s = abs(shift)
        pad = x[-s:][::-1] if pad_mode == 'reflect' else np.full(s, constant_value)
        y = np.concatenate([x[s:], pad])
    return y.astype(x.dtype)


def permute_subsegments(
    x: np.ndarray,
    k_range: tuple[int, int] = (3, 6),
    min_len: int = 10,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """Corta la señal en k trozos contiguos (k en [k_min,k_max]) y los reordena."""
    rng = np.random.default_rng(rng)
    x = np.asarray(x, dtype=float)
    n = len(x)
    k = int(rng.integers(k_range[0], k_range[1] + 1))
    # elegir k-1 cortes evitando segmentos demasiado cortos
    if n < (k * min_len):
        return x  # no hay espacio suficiente
    valid_cuts = np.arange(min_len, n - min_len)
    cuts = np.sort(rng.choice(valid_cuts, size=k - 1, replace=False))
    idxs = np.concatenate([[0], cuts, [n]])
    segs = [x[idxs[i]:idxs[i + 1]] for i in range(len(idxs) - 1) if (idxs[i + 1] - idxs[i]) >= min_len]
    if len(segs) <= 1:
        return x
    order = rng.permutation(len(segs))
    return np.concatenate([segs[i] for i in order]).astype(x.dtype)


# --------- Wrapper para aplicar a clases minoritarias en un DataFrame ---------

def augment_minority_classes(
    df: pd.DataFrame,
    label_col: str = 'label',
    value_columns: list[str] | None = None,
    strategy: str = 'to_majority',  # 'to_majority' | 'ratio' | 'fixed'
    ratio: float | None = None,
    target: int | None = None,
    transforms: list | None = None,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Genera nuevos ejemplos de las clases minoritarias aplicando aleatoriamente una
    de las transformaciones definidas arriba a filas existentes (con reposición).

    - strategy='to_majority': sube minoritarias al tamaño de la mayoritaria.
    - strategy='ratio': sube minoritarias a int(majority_count * ratio).
    - strategy='fixed': sube minoritarias a 'target'.
    """
    rng = np.random.default_rng(random_state)

    if value_columns is None:
        excluded = {label_col, 'class'}
        value_columns = [c for c in df.columns if c not in excluded]

    counts = df[label_col].value_counts()
    if counts.empty or len(counts) == 1:
        return df.copy()

    majority_count = counts.max()
    if strategy == 'to_majority':
        target_count = majority_count
    elif strategy == 'ratio':
        if ratio is None or ratio <= 0:
            raise ValueError("Debes pasar ratio>0 para strategy='ratio'.")
        target_count = int(majority_count * ratio)
    elif strategy == 'fixed':
        if target is None or target <= 0:
            raise ValueError("Debes pasar target>0 para strategy='fixed'.")
        target_count = target
    else:
        raise ValueError("strategy debe ser 'to_majority' | 'ratio' | 'fixed'.")

    if not transforms:
        transforms = [
            lambda x: jitter_signal(x, sigma=0.02, rng=rng),
            lambda x: scale_amplitude(x, scale_range=(0.9, 1.1), rng=rng),
            lambda x: time_shift_signal(x, max_shift=5, pad_mode='reflect', rng=rng),
            lambda x: permute_subsegments(x, k_range=(3, 6), min_len=10, rng=rng),
        ]

    parts = [df]

    for cls, cnt in counts.items():
        if cnt >= target_count:
            continue  # no es minoritaria con respecto al objetivo
        cls_df = df[df[label_col] == cls]
        needed = target_count - cnt
        # muestrear índices base con reposición
        base_idx = cls_df.sample(n=needed, replace=True, random_state=int(rng.integers(0, 1e9))).index
        augmented_rows = []
        for idx in base_idx:
            row = df.loc[idx]
            sig = row[value_columns].to_numpy(dtype=float)
            # elegir una transformación al azar
            t = transforms[int(rng.integers(0, len(transforms)))]
            sig_aug = t(sig)
            new_row = row.copy()
            new_row[value_columns] = sig_aug
            augmented_rows.append(new_row)
        if augmented_rows:
            parts.append(pd.DataFrame(augmented_rows))

    out = pd.concat(parts, axis=0).sample(frac=1.0, random_state=int(rng.integers(0, 1e9))).reset_index(drop=True)
    return out


In [26]:
# Ejemplos de uso:
# train_data_us_os_da = augment_minority_classes(train_data_us_os, label_col='label', strategy='to_majority')
train_data_us_os_da = augment_minority_classes(train_data_us_os, label_col='label', strategy='ratio', ratio=0.05)
# train_data_us_os_da = augment_minority_classes(train_data_us_os, label_col='label', strategy='fixed', target=10000)

In [27]:
def verify_class_balance(df_before, df_after, label_col='label'):
    before = df_before[label_col].value_counts().sort_index()
    after = df_after[label_col].value_counts().sort_index()
    comp = pd.DataFrame({
        'before': before,
        'after': after,
        'delta': (after - before)
    })
    comp['before_pct'] = (comp['before'] / comp['before'].sum() * 100).round(1)
    comp['after_pct']  = (comp['after']  / comp['after'].sum()  * 100).round(1)
    print(comp.fillna(0).astype({'before':'int64','after':'int64','delta':'int64'}))

# Ejemplos:
verify_class_balance(train_data, train_data_us_os_da)   # oversampling / augmentación



                                   before  after  delta  before_pct  after_pct
label                                                                         
Artial Premature                     2223   3500   1277         2.5        6.5
Fusion of paced and normal           6431   6431      0         7.3       11.9
Fusion of ventricular and normal      641   3500   2859         0.7        6.5
Normal                              72471  35000 -37471        82.8       64.6
Premature ventricular contraction    5788   5788      0         6.6       10.7


# Entrenamientos

* Sampler ponderado (WeightedRandomSampler) para batches balanceados.
* Regularización (dropout, weight decay) para evitar overfit por oversampling.
* Class weights (https://discuss.pytorch.org/t/passing-the-weights-to-crossentropyloss-correctly/14731)

In [28]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [29]:
# definimos el dispositivo que vamos a usar
DEVICE = "cpu"  # por defecto, usamos la CPU
if torch.cuda.is_available():
    DEVICE = "cuda"  # si hay GPU, usamos la GPU
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # si no hay GPU, pero hay MPS, usamos MPS
elif torch.xpu.is_available():
    DEVICE = "xpu"  # si no hay GPU, pero hay XPU, usamos XPU

print(f"Usando {DEVICE}")

NUM_WORKERS = 0 # Win y MacOS pueden tener problemas con múltiples workers
if sys.platform == 'linux':
    NUM_WORKERS = 4  # numero de workers para cargar los datos (depende de cada caso)

print(f"Usando {NUM_WORKERS}")

Usando cpu
Usando 0


In [30]:
BATCH_SIZE = 256  # tamaño del batch

## Data Loader

In [31]:
class CardiacLogDataset(Dataset):
    def __init__(self, sequences, labels, transforms=None):
        # Convert to numpy arrays so positional indexing works
        self.sequences = sequences.to_numpy(dtype=float) if hasattr(sequences, "to_numpy") else np.asarray(sequences, dtype=float)
        self.labels = labels.to_numpy(dtype=int) if hasattr(labels, "to_numpy") else np.asarray(labels, dtype=int)
        self.transforms = transforms

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.transforms:
            sequence = self.transforms(sequence)
        return sequence, label

In [32]:
from sklearn.model_selection import train_test_split

train_labels = train_data_us_os_da['class'] 
train_data = train_data_us_os_da.drop(columns=['class','label','duracion_no_ceros'])

# División de los datos en entrenamiento (train) y validación (val)
train_data, val_data, train_labels, val_labels = train_test_split(
    train_data, 
    train_labels, 
    test_size=0.2, 
    random_state=SEED, 
    stratify=train_labels
)


In [33]:
train_data

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186
576,1.000000,0.798343,0.624309,0.417127,0.306630,0.204420,0.104972,0.069061,0.055249,0.055249,0.063536,0.055249,0.066298,0.055249,0.077348,0.063536,0.058011,0.055249,0.060773,0.044199,0.060773,0.041436,0.055249,0.038674,0.074586,0.063536,0.080110,0.085635,0.127072,0.129834,0.171271,0.190608,0.237569,0.220994,0.248619,0.229282,0.243094,0.215470,0.215470,0.198895,0.198895,0.168508,0.182320,0.160221,0.168508,0.168508,0.176796,0.160221,0.157459,0.157459,0.176796,0.171271,0.179558,0.165746,0.185083,0.162983,0.182320,0.171271,0.182320,0.168508,0.168508,0.162983,0.162983,0.146409,0.171271,0.176796,0.223757,0.223757,0.248619,0.262431,0.287293,0.287293,0.279006,0.240331,0.160221,0.107735,0.118785,0.107735,0.127072,0.121547,0.129834,0.132597,0.146409,0.116022,0.116022,0.143646,0.364641,0.698895,0.950276,0.906077,0.693370,0.450276,0.364641,0.179558,0.088398,0.024862,0.024862,0.011050,0.011050,0.011050,0.000000,0.008287,0.008287,0.000000,0.013812,0.000000,0.022099,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7304,1.000000,0.590226,0.582707,0.582707,0.578947,0.563910,0.537594,0.466165,0.308271,0.187970,0.000000,0.060150,0.112782,0.120301,0.124060,0.221805,0.195489,0.191729,0.184211,0.218045,0.206767,0.225564,0.218045,0.221805,0.214286,0.214286,0.203008,0.214286,0.195489,0.214286,0.199248,0.221805,0.225564,0.244361,0.244361,0.278196,0.281955,0.312030,0.312030,0.342105,0.342105,0.364662,0.360902,0.375940,0.330827,0.342105,0.319549,0.323308,0.312030,0.330827,0.323308,0.323308,0.296992,0.319549,0.289474,0.285714,0.263158,0.270677,0.248120,0.255639,0.236842,0.259398,0.229323,0.236842,0.218045,0.236842,0.221805,0.236842,0.210526,0.221805,0.191729,0.214286,0.195489,0.218045,0.203008,0.214286,0.203008,0.203008,0.184211,0.214286,0.191729,0.221805,0.195489,0.214286,0.187970,0.199248,0.203008,0.214286,0.187970,0.214286,0.206767,0.221805,0.229323,0.229323,0.210526,0.251880,0.248120,0.327068,0.748120,0.921053,0.571429,0.590226,0.582707,0.597744,0.552632,0.571429,0.526316,0.439850,0.300752,0.142857,0.056391,0.116541,0.071429,0.109023,0.146617,0.191729,0.154135,0.154135,0.157895,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26819,1.000000,0.701493,0.160448,0.078358,0.134328,0.130597,0.093284,0.070896,0.067164,0.070896,0.067164,0.052239,0.059701,0.078358,0.078358,0.070896,0.078358,0.093284,0.093284,0.082090,0.085821,0.100746,0.089552,0.089552,0.093284,0.108209,0.111940,0.100746,0.111940,0.130597,0.126866,0.115672,0.149254,0.175373,0.186567,0.167910,0.164179,0.16791

In [34]:
test_labels = test_data['class'] 
test_data = test_data.drop(columns=['class','label'])


In [35]:
train_dataset = CardiacLogDataset(train_data, train_labels)
val_dataset   = CardiacLogDataset(val_data,   val_labels)
test_dataset  = CardiacLogDataset(test_data,  test_labels)

In [36]:
def get_data_loaders(batch_size):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader, test_loader


In [37]:
train_loader, val_loader, test_loader = get_data_loaders(BATCH_SIZE)

In [38]:
class_counts = Counter(train_labels)
total = sum(class_counts.values())
class_weights = {cls: total/count for cls, count in class_counts.items()}
class_weights = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float).to(DEVICE)

print(f"Class Weights: {class_weights}")

Class Weights: tensor([ 1.5491, 15.4911])


In [39]:
class_counts.values()

dict_values([28000, 5145, 2800, 4630, 2800])

## RNN

In [55]:
CRITERION = nn.CrossEntropyLoss()  # función de pérdida con pesos para cada clase
LR = 0.0005
EPOCHS = 150

In [56]:
# Constantes
SEQUENCE_LENGTH = 120  
PADDING_VALUE = 0  # Padding value

# Example cardiac log sequence (substitute with real data)
# Ensure the sequence length matches your data
# data_indices = torch.tensor([
#     [0, 1, 2, 3, 4, 5] + [0] * (SEQUENCE_LENGTH - 6)
# ], dtype=torch.long)  # Ensure dtype is long for embedding

# Apply embedding
# embedded = embedding_layer(data_indices)

# Print results
# print(f"Data indices (first sequence): {data_indices[0]}")
# print(f"Embedded shape: {embedded.shape}")
# print(f"Embedded (first sequence): {embedded[0]}")

### Weight and Bias

In [57]:
import wandb

In [58]:
wandb.login(key="371d1ec6f09f8bd9b3f43d38d319dae857ea940f")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Matías\_netrc


True

### Modelo

In [59]:
# from torchinfo import summary

# class CardiacRNNClassifier(nn.Module):
#     def __init__(self, hidden_dim, num_classes=10, rnn_type='rnn', bidirectional=False, input_dim=1):
#         super().__init__()
#         self.bidirectional = bidirectional
#         hidden_out = hidden_dim * (2 if bidirectional else 1)
#         if rnn_type == 'lstm':
#             self.rnn = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=bidirectional)
#         elif rnn_type == 'gru':
#             self.rnn = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=bidirectional)
#         else:
#             self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=bidirectional)
        
#         # Capa final para clasificacion
#         self.fc = nn.Linear(hidden_out, num_classes)
    
#     def forward(self, x):
#         if x.ndim == 2:
#             x = x.unsqueeze(-1)
#         x = x.float()
#         out, hidden = self.rnn(x)
#         if isinstance(hidden, tuple):  # LSTM
#             hidden = hidden[0]         # h_n
#         # hidden: (num_layers * num_directions, B, hidden_dim)
#         if self.bidirectional:
#             x = torch.cat((hidden[-2], hidden[-1]), dim=1)  # (B, hidden_dim*2)
#         else:
#             x = hidden[-1]                                   # (B, hidden_dim)
#         return self.fc(x)



In [60]:
import torch.nn.functional as F  # agregar este import

class CardiacRNNClassifier(nn.Module):
    def __init__(self, hidden_dim, num_classes=10, rnn_type='rnn', bidirectional=False, input_dim=1, seq_len=120):
        super().__init__()
        self.bidirectional = bidirectional
        self.seq_len = seq_len
        hidden_out = hidden_dim * (2 if bidirectional else 1)
        if rnn_type == 'lstm':
            self.rnn = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=bidirectional)
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=bidirectional)
        else:
            self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=bidirectional)
        
        # Capa final para clasificacion
        self.fc = nn.Linear(hidden_out, num_classes)
    
    def forward(self, x):
        if x.ndim == 2:
            x = x.unsqueeze(-1)  # (B, T) -> (B, T, 1)
        x = x.float()

        # Truncado/padding a longitud fija self.seq_len
        T = x.size(1)
        if T > self.seq_len:
            x = x[:, :self.seq_len, :]  # truncar a los primeros seq_len pasos
        elif T < self.seq_len:
            pad_t = self.seq_len - T
            x = F.pad(x, (0, 0, 0, pad_t))  # pad en dimensión temporal (derecha)

        out, hidden = self.rnn(x)
        if isinstance(hidden, tuple):  # LSTM
            hidden = hidden[0]         # h_n
        # hidden: (num_layers * num_directions, B, hidden_dim)
        if self.bidirectional:
            x = torch.cat((hidden[-2], hidden[-1]), dim=1)  # (B, hidden_dim*2)
        else:
            x = hidden[-1]                                   # (B, hidden_dim)
        return self.fc(x)

In [61]:
from torchinfo import summary

summary(CardiacRNNClassifier(hidden_dim=256, rnn_type='rnn', bidirectional=False, input_dim=1, seq_len=SEQUENCE_LENGTH),
input_size=(BATCH_SIZE, SEQUENCE_LENGTH),
dtypes=[torch.int32],
)

Layer (type:depth-idx)                   Output Shape              Param #
CardiacRNNClassifier                     [256, 10]                 --
├─RNN: 1-1                               [256, 120, 256]           66,304
├─Linear: 1-2                            [256, 10]                 2,570
Total params: 68,874
Trainable params: 68,874
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 2.04
Input size (MB): 0.12
Forward/backward pass size (MB): 62.94
Params size (MB): 0.28
Estimated Total Size (MB): 63.33

## Weight and Bias

In [62]:
WANDB_PROJECT = 'Entrega 2 - RNN, LSTM, GRU'

In [63]:
def train_wandb_rnn(config=None):
    import os
    import numpy as np
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score

    AVERAGE = "weighted"

    def _as_int(x, default=None):
        if x is None:
            return default
        if isinstance(x, (list, tuple)):
            return int(x[0]) if len(x) > 0 else default
        try:
            import torch
            if isinstance(x, torch.Tensor):
                return int(x.item())
        except Exception:
            pass
        try:
            return int(x)
        except Exception:
            return default

    with wandb.init(config=config):
        config = wandb.config

        # Normalizar y validar rnn_type
        rnn_type = getattr(config, "rnn_type", "rnn").lower()
        assert rnn_type in {"rnn", "lstm", "gru"}, f"rnn_type no soportado: {rnn_type}"

        # Preparar loaders con el batch_size
        train_loader, val_loader, _ = get_data_loaders(
            _as_int(getattr(config, "batch_size", 64), default=64)
        )

        # Crear el modelo CardiacRNNClassifier según el sweep
        num_classes = _as_int(getattr(config, "num_classes", 10), default=10)
        model_kwargs = dict(
            hidden_dim=_as_int(config.hidden_dim),
            num_classes=num_classes,
            rnn_type=rnn_type,
            bidirectional=bool(getattr(config, "bidirectional", False)),
            input_dim=1,
        )
        if hasattr(config, "seq_len"):
            model_kwargs["seq_len"] = _as_int(getattr(config, "seq_len", 120), default=120)

        model = CardiacRNNClassifier(**model_kwargs).to(DEVICE)

        # Configurar optimizador con weight decay
        if config.optimizer == "adam":
            optimizer = optim.Adam(
                model.parameters(),
                lr=float(config.learning_rate),
                weight_decay=float(config.weight_decay)
            )
        else:
            optimizer = optim.SGD(
                model.parameters(),
                lr=float(config.learning_rate),
                momentum=0.9,
                weight_decay=float(config.weight_decay)
            )

        # Registrar el modelo en wandb
        wandb.watch(model, log="all", log_freq=100)

        # Entrenamiento
        best_val_loss = float("inf")
        best_val_acc = 0.0

        for epoch in range(EPOCHS):
            # Training phase
            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0

            # Acumuladores para métricas
            train_preds = []
            train_targs = []

            for inputs, targets in train_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = CRITERION(outputs, targets)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = outputs.max(1)
                train_total += targets.size(0)
                train_correct += predicted.eq(targets).sum().item()

                # Guardar para métricas por-época
                train_preds.append(predicted.detach().cpu())
                train_targs.append(targets.detach().cpu())

            train_loss = train_loss / len(train_loader)
            train_acc = 100. * train_correct / train_total

            # Métricas de train (weighted)
            train_preds = torch.cat(train_preds).numpy()
            train_targs = torch.cat(train_targs).numpy()
            train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(
                train_targs, train_preds, average=AVERAGE, zero_division=0
            )
            train_precision *= 100.0
            train_recall *= 100.0
            train_f1 *= 100.0

            # Validation phase
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0

            val_preds = []
            val_targs = []

            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                    outputs = model(inputs)
                    loss = CRITERION(outputs, targets)

                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    val_total += targets.size(0)
                    val_correct += predicted.eq(targets).sum().item()

                    # Guardar para métricas por-época
                    val_preds.append(predicted.detach().cpu())
                    val_targs.append(targets.detach().cpu())

            val_loss = val_loss / len(val_loader)
            val_acc = 100. * val_correct / val_total

            # Métricas de validación (weighted)
            val_preds = torch.cat(val_preds).numpy()
            val_targs = torch.cat(val_targs).numpy()
            val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
                val_targs, val_preds, average=AVERAGE, zero_division=0
            )
            val_precision *= 100.0
            val_recall *= 100.0
            val_f1 *= 100.0

            # Logging
            wandb.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "train_precision": train_precision,
                "train_recall": train_recall,
                "train_f1": train_f1,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "val_precision": val_precision,
                "val_recall": val_recall,
                "val_f1": val_f1,
                "learning_rate": optimizer.param_groups[0]['lr'],
                "rnn_type": rnn_type,
                "seq_len": model_kwargs.get("seq_len"),
            })

            # Guardar mejor modelo (evitar symlink en Windows)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_val_loss = val_loss
                save_path = os.path.join(wandb.run.dir, "rnn_best_model.pth")
                torch.save(model.state_dict(), save_path)

            # Print progress
            print(f'Epoch: {epoch+1}/{EPOCHS}')
            print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Precision: {train_precision:.2f}% Recall: {train_recall:.2f}% F1: {train_f1:.2f}%')
            print(f'Val   Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%   | Precision: {val_precision:.2f}% Recall: {val_recall:.2f}% F1: {val_f1:.2f}%\n')

        # Log final metrics
        wandb.run.summary.update({
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "weight_decay": float(config.weight_decay),
            "batch_size": _as_int(getattr(config, "batch_size", 64), default=64),
            "hidden_dim": _as_int(config.hidden_dim),
            "bidirectional": bool(getattr(config, "bidirectional", False)),
            "rnn_type": rnn_type,
            "seq_len": model_kwargs.get("seq_len"),
            "num_classes": num_classes,
        })

### RNN

In [64]:
sweep_config = {
    'name': 'RNN-ECG',
    "method": "bayes",  
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "rnn_type": {"value": "rnn"},
        "hidden_dim": {"values": [64, 128, 256, 512]},
        "bidirectional": {"values": [False, True]},
        "batch_size": {"values": [32, 64, 128]},
        "optimizer": {"values": ["adam", "sgd"]},
        "learning_rate": {"distribution": "uniform", "min": 0.0001, "max": 0.001},
        "weight_decay": {"values": [0.0, 1e-5, 1e-4, 1e-3]},
        "seq_len": {"values": [80,120,130,140,150,160]} 
    }
}

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)

Create sweep with ID: 48ck289w
Sweep URL: https://wandb.ai/RoJo/Entrega%202%20-%20RNN%2C%20LSTM%2C%20GRU/sweeps/48ck289w


In [65]:
wandb.agent(sweep_id, train_wandb_rnn, count=10)  

wandb: Agent Starting Run: guau96s1 with config:
wandb: 	batch_size: 32
wandb: 	bidirectional: False
wandb: 	hidden_dim: 64
wandb: 	learning_rate: 0.000869273886580118
wandb: 	optimizer: adam
wandb: 	rnn_type: rnn
wandb: 	seq_len: 120
wandb: 	weight_decay: 0.001


Epoch: 1/150
Train Loss: 1.1477 | Train Acc: 64.44% | Precision: 42.30% Recall: 64.44% F1: 50.64%
Val   Loss: 1.1297 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.1302 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1291 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1294 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1399 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.0696 | Train Acc: 64.75% | Precision: 47.98% Recall: 64.75% F1: 51.85%
Val   Loss: 1.1254 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.1282 | Train Acc: 64.56% | Precision: 52.35% Recall: 64.56% F1: 50.67%
Val   Loss: 1.1244 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.0763 | Train Acc: 64.53% | Precision: 43.74% Recall: 

epoch,▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇████
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▁▁▁▁▁▁▃▁▁▁▂▂▂▃▂▁▁▁▁▁▂▁▁▂▃▆▇███▃▁▁▁▁▁▂▁
train_f1,▁▁▁▁▁▂▃▁▁▁▆▆▂▆▁▁▁▁▁▄▇██▇██▁▃▂▁▁▁▁▁▁▂▂▁▁▂
train_loss,▇▇▇███▇███▇▇▆▇▆▅██▇▇▆▅▂▁▆▇████▇▆▇█▇▇▆██▇
train_precision,▃▂▂▃▃▁▁▃▃▃▁▁▃▃▇▅▁▄▁▄▃▃▂▆▃▇█▅▁▁▄▁▁▂▁▃▃▃▃▄
train_recall,▁▁▁▁▁▁▁▁▂▃▁▁▁▁▅▁▁▁▁▂▃▃▁▄▆█▆▃▁▁▁▁▁▁▁▂▂▁▁▁
val_acc,▂▂▂▂▂▂▂▃▂▂▂▂▄▃▇▂▂▂▂▂▁▆█▇█▃▂▂▂▂▂▂▂▂▂▃▂▄▂▂
val_f1,▁▁▁▁▁▁▁▁▁▂▃▃▁▁▁▂▅▆▁▁▁▃▁▁▂█▇▂▂▁▁▁▁▁▂▁▁▁▁▁
+3,...


wandb: Agent Starting Run: xjih2nwp with config:
wandb: 	batch_size: 128
wandb: 	bidirectional: False
wandb: 	hidden_dim: 64
wandb: 	learning_rate: 0.0005745253723348714
wandb: 	optimizer: sgd
wandb: 	rnn_type: rnn
wandb: 	seq_len: 120
wandb: 	weight_decay: 0.0001


Epoch: 1/150
Train Loss: 1.4548 | Train Acc: 63.39% | Precision: 41.64% Recall: 63.39% F1: 50.26%
Val   Loss: 1.1929 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.1672 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1519 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1463 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1418 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.1398 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1373 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.1363 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1350 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.1345 | Train Acc: 64.55% | Precision: 41.67% Recall: 

...

wandb: ERROR Error while calling W&B API: context deadline exceeded (<Response [500]>)


Epoch: 110/150
Train Loss: 0.5712 | Train Acc: 82.46% | Precision: 81.34% Recall: 82.46% F1: 79.93%
Val   Loss: 0.5428 | Val Acc: 83.76%   | Precision: 83.18% Recall: 83.76% F1: 81.70%



wandb: ERROR Error while calling W&B API: An internal error occurred. Please contact support. (<Response [500]>)
wandb: ERROR Error while calling W&B API: An internal error occurred. Please contact support. (<Response [500]>)
wandb: Network error (HTTPError), entering retry loop.


Epoch: 111/150
Train Loss: 0.6836 | Train Acc: 77.73% | Precision: 76.62% Recall: 77.73% F1: 73.78%
Val   Loss: 0.5725 | Val Acc: 81.56%   | Precision: 81.46% Recall: 81.56% F1: 78.21%

Epoch: 112/150
Train Loss: 0.5356 | Train Acc: 83.53% | Precision: 82.72% Recall: 83.53% F1: 81.55%
Val   Loss: 0.5389 | Val Acc: 83.22%   | Precision: 82.03% Recall: 83.22% F1: 81.13%

Epoch: 113/150
Train Loss: 0.5203 | Train Acc: 83.91% | Precision: 83.22% Recall: 83.91% F1: 82.39%
Val   Loss: 0.4909 | Val Acc: 85.25%   | Precision: 84.93% Recall: 85.25% F1: 84.16%

Epoch: 114/150
Train Loss: 0.4849 | Train Acc: 85.27% | Precision: 84.88% Recall: 85.27% F1: 84.19%
Val   Loss: 0.5057 | Val Acc: 83.36%   | Precision: 83.26% Recall: 83.36% F1: 82.46%

Epoch: 115/150
Train Loss: 0.4746 | Train Acc: 85.64% | Precision: 85.32% Recall: 85.64% F1: 84.65%
Val   Loss: 0.4420 | Val Acc: 86.70%   | Precision: 86.51% Recall: 86.70% F1: 85.69%

Epoch: 116/150
Train Loss: 0.4615 | Train Acc: 85.97% | Precision: 85.

epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▆▆▆▇▇▇▇▇▇███████▅▂▆▇▇▃▄▆▆▆▆▆▆▃▄▆▄▂▄▅▅▅
train_f1,▃▃▄▅▅▇▇▇▇▇▇▇▇██▇▇▇█████▂▃▄▆▇▇▇▇▄▅▄▁▄▄▅▅▅
train_loss,▆▆▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▃▂▂▁▁▁▅▇▃▆▅▄▄▃▃▃▅▃▅█▆▅▄
train_precision,▁▁▃▅▅▆▇▇▇▇▇▇▇█▇█████▇▇███████▃▅▅▅▆▇▇▇▇▄▆
train_recall,▁▁▄▄▆▆▆▇▇▇▇▇▇▇▇▇▇█████████▄▂▃▆▃▆▆▅▆▃▄▄▅▅
val_acc,▁▃▅▆▆▇▇▆▇▇▇▇▇▅▅▇▇▇███▇███▄▅▆▇▄▆▆▆▅▆▁▂▂▂▅
val_f1,▁▃▃▄▅▇▇▇▇▇█▄▅▆▇██▇██████▅▇▇▅▆▇▇▇▆▃▅▁▅▅▅▅
+3,...


### LSTM

In [68]:
sweep_config = {
    "name": "LSTM-ECG",
    "method": "bayes",  
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "rnn_type": {"value": "lstm"},
        "hidden_dim": {"values": [64, 128, 256, 512]},
        "bidirectional": {"values": [False, True]},
        "batch_size": {"values": [32, 64, 128]},
        "optimizer": {"values": ["adam", "sgd"]},
        "learning_rate": {"distribution": "uniform", "min": 0.0001, "max": 0.001},
        "weight_decay": {"values": [0.0, 1e-5, 1e-4, 1e-3]},
        "seq_len": {"values": [80,120,130,140,150,160]}
    }
}

In [69]:
wandb.agent(sweep_id, train_wandb_rnn, count=3) 

wandb: Agent Starting Run: xzcgji1e with config:
wandb: 	batch_size: 64
wandb: 	bidirectional: True
wandb: 	hidden_dim: 512
wandb: 	learning_rate: 0.00011416583404421928
wandb: 	optimizer: sgd
wandb: 	rnn_type: rnn
wandb: 	seq_len: 140
wandb: 	weight_decay: 0.0001


Epoch: 1/150
Train Loss: 1.6671 | Train Acc: 62.61% | Precision: 45.70% Recall: 62.61% F1: 50.19%
Val   Loss: 1.2498 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.1810 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1464 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1381 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1332 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.1310 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1292 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.1281 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1277 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.1264 | Train Acc: 64.55% | Precision: 41.67% Recall: 

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████
train_f1,▁▁▁▁▁▃▃▃▃▃▄▄▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇█████████████
train_loss,██████▇▆▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
train_precision,▂▁▁▁▁▁▁▁▁▁▂▁▃▃▃▄▆▇▇▇▇▇▇▇▇▇▇█████████████
train_recall,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████
val_acc,▁▁▁▁▁▁▁▁▃▃▃▃▄▄▅▅▆▇▇▇▇▇▇▇▇▇▇█▇█▇█████████
val_f1,▁▁▁▁▁▁▃▃▃▃▄▅▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇███████████
+3,...


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: jj4276a1 with config:
wandb: 	batch_size: 64
wandb: 	bidirectional: True
wandb: 	hidden_dim: 512
wandb: 	learning_rate: 0.00010277632970153458
wandb: 	optimizer: sgd
wandb: 	rnn_type: rnn
wandb: 	seq_len: 140
wandb: 	weight_decay: 0.0001


Epoch: 1/150
Train Loss: 1.7028 | Train Acc: 62.35% | Precision: 43.57% Recall: 62.35% F1: 50.62%
Val   Loss: 1.2823 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.2103 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1615 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1459 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1367 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.1334 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1310 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.1294 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1281 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.1273 | Train Acc: 64.55% | Precision: 41.67% Recall: 

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇██
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▁▁▁▃▃▃▃▃▃▃▃▄▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
train_f1,▁▁▁▁▁▁▁▁▁▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇▇█████████▇█▇███
train_loss,█▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁
train_precision,▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▄▄▄▅▅▅▆▆▇▇▇▇██████████▆██
train_recall,▁▁▁▁▁▁▂▃▃▃▃▃▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇██████████▇▇█
val_acc,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▄▄▆▇▇▇▇▇▇▇▇██████▆▇▇▇▇
val_f1,▁▁▁▁▁▁▁▁▁▁▃▃▃▄▄▅▅▆▇▇▇▇▇▇▇▇██████████▇█▇█
+3,...


wandb: Agent Starting Run: tb6h0zht with config:
wandb: 	batch_size: 64
wandb: 	bidirectional: True
wandb: 	hidden_dim: 512
wandb: 	learning_rate: 0.0003786721020285109
wandb: 	optimizer: sgd
wandb: 	rnn_type: rnn
wandb: 	seq_len: 140
wandb: 	weight_decay: 0.001


Epoch: 1/150
Train Loss: 1.3460 | Train Acc: 64.10% | Precision: 42.94% Recall: 64.10% F1: 50.62%
Val   Loss: 1.1324 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.1254 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1214 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1180 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1146 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.1090 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1034 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.0922 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.0778 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.0246 | Train Acc: 65.75% | Precision: 53.67% Recall: 

epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▃▃▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████████
train_f1,▁▃▄▄▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████████████
train_loss,██▇▆▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_precision,▁▃▃▃▄▆▆▇▇▇▇▇▇▇▇█████████████████████████
train_recall,▁▃▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████████
val_acc,▁▂▃▃▃▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇███████▇████████████
val_f1,▁▁▂▄▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇███████████████████
+3,...


### GRU

In [70]:
sweep_config = {
    "name": "GRU-ECG",
    "method": "bayes", 
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "rnn_type": {"value": "gru"},
        "hidden_dim": {"values": [64, 128, 256, 512]},
        "bidirectional": {"values": [False, True]},
        "batch_size": {"values": [32, 64, 128]},
        "optimizer": {"values": ["adam", "sgd"]},
        "learning_rate": {"distribution": "uniform", "min": 0.0001, "max": 0.001},
        "weight_decay": {"values": [0.0, 1e-5, 1e-4, 1e-3]},
        "seq_len": {"values": [80,120,130,140,150,160]},
    }
}

In [71]:
wandb.agent(sweep_id, train_wandb_rnn, count=3) 

wandb: Agent Starting Run: mcep5rew with config:
wandb: 	batch_size: 128
wandb: 	bidirectional: True
wandb: 	hidden_dim: 512
wandb: 	learning_rate: 0.00022152074066599935
wandb: 	optimizer: sgd
wandb: 	rnn_type: rnn
wandb: 	seq_len: 140
wandb: 	weight_decay: 0.001


Epoch: 1/150
Train Loss: 1.7225 | Train Acc: 61.70% | Precision: 42.14% Recall: 61.70% F1: 50.04%
Val   Loss: 1.2989 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.2288 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1764 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1523 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1380 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.1325 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1295 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.1273 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1261 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.1247 | Train Acc: 64.55% | Precision: 41.67% Recall: 

epoch,▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇███
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▁▁▁▂▃▃▃▃▃▄▄▄▅▆▆▆▇▇▅▇▇▇▇▇▇█▇███████████
train_f1,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▄▄▄▆▇▇▇▇▇▇▇▇████████████
train_loss,███████▇▇▇▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
train_precision,▁▁▁▁▁▁▃▃▃▃▃▃▄▄▄▄▅▆▆▆▇▇▇▇████████████████
train_recall,▁▁▁▁▁▁▁▁▁▂▃▃▃▃▄▅▅▆▆▆▆▇▅▇▇▇▇▇▇▇██████████
val_acc,▁▁▁▁▁▁▁▃▃▃▃▃▃▄▄▆▇▇▇▆▇▇▇▇▇▇█▇▇▇██████████
val_f1,▁▁▁▁▁▁▃▃▃▃▃▃▄▆▇▇▇▇▇▇▇▇▇█▇███████████████
+3,...


wandb: Agent Starting Run: iqb627zz with config:
wandb: 	batch_size: 64
wandb: 	bidirectional: True
wandb: 	hidden_dim: 512
wandb: 	learning_rate: 0.0003428441737510262
wandb: 	optimizer: sgd
wandb: 	rnn_type: rnn
wandb: 	seq_len: 130
wandb: 	weight_decay: 0.0001


Epoch: 1/150
Train Loss: 1.3569 | Train Acc: 63.42% | Precision: 43.06% Recall: 63.42% F1: 50.71%
Val   Loss: 1.1393 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.1303 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1281 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1247 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1237 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.1216 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1203 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.1184 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1185 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.1141 | Train Acc: 64.55% | Precision: 41.67% Recall: 

epoch,▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▁▃▃▄▄▄▆▆▇▇▇▇▇██████████████▇██▅▆▇▇▇▇▇▇
train_f1,▁▁▁▁▃▃▃▄▆▇▇▇▇▇▇▇▇████████████▇██▄▅▇▇▇▇▇▇
train_loss,██▆▆▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▃▂▁▁▁▁▄▃▂▂
train_precision,▁▃▄▅▅▇▇▇▇▇▇▇▇▇████████████▇▇███████▇▇▇▇█
train_recall,▁▁▁▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇█████████▆▇█████▆▅▆▇▇▇
val_acc,▁▁▃▃▃▄▄▄▆▆▇▆▇▇▇▇▇▇▇▇█████████▇█████▆▃▆▇▇
val_f1,▁▁▁▁▃▃▄▇▇▇▇▇▇▇█▇██▇██████████▇█▇▇██▆▇▇▇█
+3,...


wandb: Agent Starting Run: pzkjib2f with config:
wandb: 	batch_size: 64
wandb: 	bidirectional: True
wandb: 	hidden_dim: 512
wandb: 	learning_rate: 0.000311396794147487
wandb: 	optimizer: sgd
wandb: 	rnn_type: rnn
wandb: 	seq_len: 150
wandb: 	weight_decay: 0.001


Epoch: 1/150
Train Loss: 1.4004 | Train Acc: 63.45% | Precision: 41.67% Recall: 63.45% F1: 50.30%
Val   Loss: 1.1423 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 2/150
Train Loss: 1.1313 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1262 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 3/150
Train Loss: 1.1245 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1243 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 4/150
Train Loss: 1.1217 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1210 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 5/150
Train Loss: 1.1184 | Train Acc: 64.55% | Precision: 41.67% Recall: 64.55% F1: 50.65%
Val   Loss: 1.1194 | Val Acc: 64.55%   | Precision: 41.67% Recall: 64.55% F1: 50.65%

Epoch: 6/150
Train Loss: 1.1151 | Train Acc: 64.55% | Precision: 41.67% Recall: 

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
seq_len,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▁▁▂▃▃▃▄▆▇▇▇▇▇█▇██████▁▁▃▄▄▅▅▅▅▅▄▅▅▆▆▆▆
train_f1,▁▁▁▁▂▃▄▆▇▇▇▇▇▇▇████████████▃▃▄▅▅▅▅▅▃▅▆▇▇
train_loss,██▆▆▆▅▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▇▄▄▅▄▅▄▄▇▅▄▄▄▄▄▃
train_precision,▁▁▁▁▃▅▅▆▇▇▇▇▇▇▇▇▇███████████▂▄▅▆▅▅▆▇▇▇▇▇
train_recall,▁▁▁▃▃▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇█████▁▁▂▂▄▅▂▃▄▅▃▅▆▆▆
val_acc,▁▁▁▁▃▃▄▆▆▇▇▇▇▇▇▇█▇████████▁▁▂▃▅▅▅▄▄▅▅▆▆▆
val_f1,▁▁▁▁▃▅▄▅▆▇▇▇▇▇█▇███████▁▂▂▃▃▃▄▅▅▆▅▄▅▅▆▅▇
+3,...
